# Add CPS parameters to Extended IBTrACS

In [1]:
# Setup environment
import huracanpy
import xarray as xr
import numpy as np

from tqdm import tqdm

In [2]:
# Script parameters
## Extended IBTrACS file you want to add the attributes to (may be the minimal one or one you already added info to)
IN_FILE = "../files/extended_ibtracs.nc"
## Output file automatically determined. You may change if you like.
OUT_FILE = IN_FILE[:-3]+"_CPS"+".nc"

In [3]:
# Open extended IBTrACS
eib = xr.open_dataset(IN_FILE)
#eib["record"] = eib.record # Explicit record as a coordinate

In [4]:
# Open the datasets with the CPS parameters
datasets = ["ERA5", "JRA3Q"]
tracks_with_CPS = {}
for ds in datasets:
    tracks_with_CPS["TRACK-"+ds] = huracanpy.load("../input/"+ds+"_TRACK_NATL_tcident_nolat_CPS.nc")

In [5]:
L = []
for ds in tqdm(tracks_with_CPS):
    # Transform the datasets into dataframes
    t_CPS_df = tracks_with_CPS[ds].to_dataframe()
    teib_df = eib.sel(dataset = ds).squeeze().reset_coords().to_dataframe()
    for var in ["lon", "lat"]: # Round up the coordinates for the matching to work
        N = 0
        t_CPS_df[var] = np.round(t_CPS_df[var], 0)
        teib_df[var] = np.round(teib_df[var], 0)
    # Merge extended ib and CPS dataset on lon, lat and time
    M = teib_df[["lon", "lat", "time",]].reset_index().merge(t_CPS_df[["lon", "lat", "time", "Vtl", "Vtu", "B"]], how = "left", on = ["lon", "lat", "time"])
    assert len(M) == len(teib_df) # Check that no duplicate were created
    M_xr = M.set_index("record").to_xarray().assign(dataset = ds).set_coords("dataset")[["Vtu", "Vtl", "B"]] # convert back to dataset and format
    L.append(M_xr)
CPS = xr.concat(L, dim = "dataset") # Merge the CPS variables over datasets
eib_with_CPS = eib.merge(CPS) # Merge back into the full extended ibtracs dataset

100%|████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  5.91it/s]


In [6]:
# Visualise
eib_with_CPS

<xarray.Dataset> Size: 86MB
Dimensions:   (dataset: 6, record: 188964)
Coordinates:
  * dataset   (dataset) <U17 408B 'IBTrACS' 'SyCLoPS-ERA5' ... 'TRACK-JRA3Q'
  * record    (record) int64 2MB 0 1 2 3 4 ... 188960 188961 188962 188963
Data variables:
    track_id  (record) <U13 10MB ...
    lon       (dataset, record) float64 9MB ...
    lat       (dataset, record) float64 9MB ...
    name      (record) <U13 10MB ...
    pres      (dataset, record) float64 9MB ...
    wind10    (dataset, record) float64 9MB ...
    time      (record) datetime64[ns] 2MB ...
    Vtu       (dataset, record) float64 9MB nan nan nan nan ... nan nan nan nan
    Vtl       (dataset, record) float64 9MB nan nan nan nan ... nan nan nan nan
    B         (dataset, record) float64 9MB nan nan nan nan ... nan nan nan nan

In [7]:
# Save
eib_with_CPS.to_netcdf(OUT_FILE)